# Incident Response Runbook: Command and Control - Encrypted Channel

**Tactic:** Command and Control
**Technique:** Encrypted Channel (T1573)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Encrypted Channel activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add path for custom integrations
sys.path.append(os.path.join(os.path.dirname(__file__), '..', '..'))

# Import security integrations
from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()
detection_actions = []
affected_systems = []
threat_indicators = []

# 1. Analyze encrypted channel anomalies in Splunk
print(f"\n[DETECTION] Analyzing encrypted channel anomalies...")
try:
    # Query Splunk for suspicious encrypted communications
    splunk_query = """
    index=* (sourcetype=network OR sourcetype=firewall OR sourcetype=proxy)
    | search protocol IN ("https", "ssl", "tls", "ssh", "rdp", "vnc")
    | eval risk_score = case(
        protocol=="https" AND bytes_out>2000000 AND connection_count>30, 9,
        protocol=="ssl" AND dest_port IN (443, 993, 995) AND bytes_out>1000000, 8,
        protocol=="tls" AND connection_count>50, 7,
        protocol=="ssh" AND bytes_out>500000 AND connection_count>10, 6,
        protocol=="rdp" AND dest_ip NOT IN (internal_networks), 8,
        protocol=="vnc" AND dest_ip NOT IN (internal_networks), 8,
        1==1, 3
    )
    | where risk_score >= 6
    | stats count, sum(bytes_out) as total_bytes by src_ip, dest_ip, protocol, dest_port
    | sort -risk_score
    """

    alert_results = splunk.execute_query(splunk_query)
    if alert_results and len(alert_results) > 0:
        print(f"   Found {len(alert_results)} suspicious encrypted connections")

        for alert in alert_results[:10]:  # Top 10 alerts
            affected_systems.append({
                'ip': alert.get('src_ip'),
                'protocol': alert.get('protocol'),
                'dest_ip': alert.get('dest_ip'),
                'dest_port': alert.get('dest_port'),
                'total_bytes': alert.get('total_bytes', 0),
                'connection_count': alert.get('count', 0)
            })

        detection_actions.append({
            'action': 'splunk_encrypted_analysis',
            'findings': f"{len(alert_results)} suspicious encrypted connections detected",
            'method': 'Splunk',
            'status': 'success',
            'timestamp': detection_time
        })
    else:
        print(f"   No suspicious encrypted channel activity found in recent logs")

except Exception as e:
    print(f"   Error analyzing encrypted channels: {e}")

# 2. Check CrowdStrike for encrypted channel indicators
print(f"\n[DETECTION] Checking CrowdStrike for encrypted channel indicators...")
try:
    # Look for processes making suspicious encrypted connections
    cs_indicators = crowdstrike.search_indicators({
        'indicator_type': 'encrypted_connection',
        'protocols': ['https', 'ssl', 'tls', 'ssh', 'rdp', 'vnc'],
        'time_range': '24h'
    })

    if cs_indicators:
        for indicator in cs_indicators:
            if indicator.get('risk_score', 0) >= 75:
                threat_indicators.append({
                    'type': 'encrypted_connection',
                    'value': indicator.get('connection_string'),
                    'protocol': indicator.get('protocol'),
                    'source': 'CrowdStrike',
                    'risk_score': indicator.get('risk_score')
                })

        detection_actions.append({
            'action': 'crowdstrike_encrypted_check',
            'findings': f"{len(threat_indicators)} high-risk encrypted indicators",
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': detection_time
        })
        print(f"   Found {len(threat_indicators)} high-risk encrypted channel indicators")
    else:
        print(f"   No high-risk encrypted indicators found in CrowdStrike")

except Exception as e:
    print(f"   Error checking CrowdStrike: {e}")

# 3. Analyze SSL/TLS certificate anomalies
print(f"\n[DETECTION] Analyzing SSL/TLS certificate anomalies...")
try:
    # Look for self-signed certificates or unusual certificate authorities
    cert_query = """
    index=* sourcetype=ssl OR sourcetype=tls
    | search certificate_issuer!="*trusted*" OR certificate_self_signed=true
    | eval risk_score = case(
        certificate_self_signed=true AND bytes_out>100000, 8,
        certificate_issuer=="unknown" OR certificate_issuer=="self-signed", 7,
        certificate_validity_days<30, 6,
        1==1, 3
    )
    | where risk_score >= 6
    | stats count by src_ip, certificate_subject, certificate_issuer
    """

    cert_results = splunk.execute_query(cert_query)
    if cert_results:
        for cert in cert_results:
            threat_indicators.append({
                'type': 'suspicious_certificate',
                'value': cert.get('certificate_subject'),
                'issuer': cert.get('certificate_issuer'),
                'source': 'Splunk SSL analysis',
                'risk_score': 7
            })

        detection_actions.append({
            'action': 'certificate_analysis',
            'findings': f"{len(cert_results)} suspicious certificates detected",
            'method': 'Splunk',
            'status': 'success',
            'timestamp': detection_time
        })
        print(f"   Found {len(cert_results)} suspicious SSL/TLS certificates")

except Exception as e:
    print(f"   Error analyzing certificates: {e}")

# 4. Enrich threat intelligence via MISP
print(f"\n[DETECTION] Enriching threat intelligence...")
try:
    for indicator in threat_indicators:
        misp_results = misp.search_indicators(indicator['value'])
        if misp_results:
            indicator['misp_enrichment'] = misp_results
            print(f"   Enriched indicator: {indicator['value'][:50]}...")

    detection_actions.append({
        'action': 'threat_intel_enrichment',
        'findings': f"Enriched {len([i for i in threat_indicators if 'misp_enrichment' in i])} indicators",
        'method': 'MISP',
        'status': 'success',
        'timestamp': detection_time
    })

except Exception as e:
    print(f"   Error enriching threat intelligence: {e}")

# 5. Create IRIS incident case
print(f"\n[DETECTION] Creating IRIS incident case...")
try:
    incident_data = {
        'title': 'Encrypted Channel C2 Activity Detected',
        'description': f'Detected suspicious encrypted channel usage with {len(threat_indicators)} indicators',
        'severity': 'CRITICAL',
        'tactic': 'Command and Control',
        'technique': 'Encrypted Channel (T1573)',
        'affected_systems': affected_systems,
        'threat_indicators': threat_indicators,
        'detection_time': detection_time
    }

    incident_id = iris.create_incident(incident_data)
    if incident_id:
        detection_actions.append({
            'action': 'incident_creation',
            'findings': f"Incident {incident_id} created",
            'method': 'IRIS',
            'status': 'success',
            'timestamp': detection_time
        })
        print(f"   IRIS incident case created: {incident_id}")
    else:
        print(f"   Failed to create IRIS incident case")

except Exception as e:
    print(f"   Error creating IRIS case: {e}")

print(f"\n Detection complete:")
print(f"  - Affected systems identified: {len(affected_systems)}")
print(f"  - Threat indicators found: {len(threat_indicators)}")
print(f"  - IRIS case created: {incident_id if 'incident_id' in locals() else 'N/A'}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
blocked_connections = []
disabled_accounts = []

# 1. Isolate affected systems
print(f"\n[CONTAINMENT] Isolating affected systems...")
try:
    for system in affected_systems:
        if system.get('ip'):
            # Use CrowdStrike to isolate host
            isolation_result = crowdstrike.isolate_host_by_ip(system['ip'])
            if isolation_result and isolation_result.get('status') == 'isolated':
                isolated_hosts.append(system['ip'])
                containment_actions.append({
                    'action': 'host_isolation',
                    'target': system['ip'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Isolated host: {system['ip']}")
            else:
                print(f"   Failed to isolate host: {system['ip']}")
        else:
            # Use Shuffle for network isolation
            network_isolation = shuffle.isolate_system(system.get('hostname', system['ip']))
            if network_isolation:
                isolated_hosts.append(system.get('hostname', system['ip']))
                containment_actions.append({
                    'action': 'network_isolation',
                    'target': system.get('hostname', system['ip']),
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Network isolated: {system.get('hostname', system['ip'])}")
except Exception as e:
    print(f"   Error isolating systems: {e}")

# 2. Block suspicious encrypted connections
print(f"\n[CONTAINMENT] Blocking suspicious encrypted connections...")
try:
    for indicator in threat_indicators:
        if indicator.get('type') == 'encrypted_connection':
            # Extract destination IP/port from connection string
            connection_parts = indicator['value'].split(':')
            if len(connection_parts) >= 2:
                dest_ip = connection_parts[0]
                dest_port = connection_parts[1] if len(connection_parts) > 1 else None

                # Use Shuffle to block the connection
                block_result = shuffle.block_encrypted_connection(dest_ip, dest_port)
                if block_result:
                    blocked_connections.append(f"{dest_ip}:{dest_port}")
                    containment_actions.append({
                        'action': 'connection_block',
                        'target': f"{dest_ip}:{dest_port}",
                        'method': 'Shuffle',
                        'status': 'success',
                        'timestamp': containment_time
                    })
                    print(f"   Blocked encrypted connection: {dest_ip}:{dest_port}")
except Exception as e:
    print(f"   Error blocking connections: {e}")

# 3. Disable compromised accounts
print(f"\n[CONTAINMENT] Disabling compromised accounts...")
try:
    # Identify accounts associated with suspicious encrypted activity
    suspicious_accounts = []
    for system in affected_systems:
        if system.get('ip'):
            # Query Splunk for user activity on affected systems
            user_query = f"""
            index=* src_ip="{system['ip']}"
            | search protocol IN ("ssh", "rdp", "vnc")
            | stats count by user
            | where count > 5
            | sort -count
            """
            user_results = splunk.execute_query(user_query)
            if user_results:
                suspicious_accounts.extend([u.get('user') for u in user_results])

    # Remove duplicates
    suspicious_accounts = list(set(suspicious_accounts))

    for account in suspicious_accounts[:5]:  # Limit to top 5 suspicious accounts
        # Use Shuffle to disable account
        disable_result = shuffle.disable_account(account)
        if disable_result:
            disabled_accounts.append(account)
            containment_actions.append({
                'action': 'account_disable',
                'target': account,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Disabled account: {account}")
except Exception as e:
    print(f"   Error disabling accounts: {e}")

# 4. Enable enhanced monitoring for encrypted channels
print(f"\n[CONTAINMENT] Enabling enhanced monitoring...")
try:
    # Create Splunk correlation search for encrypted channel anomalies
    correlation_search = {
        'name': f'C2_Encrypted_Channel_Monitoring_{incident_id}',
        'search': """
        index=* (sourcetype=network OR sourcetype=firewall)
        | search protocol IN ("https", "ssl", "tls", "ssh", "rdp", "vnc")
        | eval risk_score = case(
            protocol=="https" AND bytes_out>1000000 AND connection_count>20, 8,
            protocol=="ssh" AND bytes_out>200000 AND connection_count>5, 7,
            protocol=="rdp" AND dest_ip NOT IN (internal_networks), 8,
            protocol=="vnc" AND dest_ip NOT IN (internal_networks), 8,
            1==1, 3
        )
        | where risk_score >= 6
        | stats count by src_ip, dest_ip, protocol
        """,
        'alert_condition': 'count > 0',
        'alert_threshold': 1
    }

    monitoring_result = splunk.create_correlation_search(correlation_search)
    if monitoring_result:
        containment_actions.append({
            'action': 'enhanced_monitoring',
            'target': f"Correlation search {correlation_search['name']}",
            'method': 'Splunk',
            'status': 'success',
            'timestamp': containment_time
        })
        print(f"   Enhanced monitoring enabled in Splunk")

    # Enable CrowdStrike enhanced detection for encrypted channels
    cs_monitoring = crowdstrike.enable_enhanced_detection('encrypted_channel_c2')
    if cs_monitoring:
        containment_actions.append({
            'action': 'cs_enhanced_detection',
            'target': 'CrowdStrike EDR',
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': containment_time
        })
        print(f"   Enhanced detection enabled in CrowdStrike")

except Exception as e:
    print(f"   Error enabling enhanced monitoring: {e}")

print(f"\n Containment complete:")
print(f"  - Hosts isolated: {len(isolated_hosts)}")
print(f"  - Connections blocked: {len(blocked_connections)}")
print(f"  - Accounts disabled: {len(disabled_accounts)}")
print(f"  - Enhanced monitoring: ")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []
terminated_processes = []
deleted_files = []
reset_credentials = []

# 1. Terminate malicious processes making encrypted connections
print(f"\n[ERADICATION] Terminating malicious processes...")
try:
    for system in affected_systems:
        if system.get('ip'):
            # Get processes associated with suspicious encrypted connections
            suspicious_processes = crowdstrike.get_processes_by_network(system['ip'], ['https', 'ssl', 'tls', 'ssh', 'rdp', 'vnc'])
            if suspicious_processes:
                for process in suspicious_processes:
                    if process.get('encrypted_connections'):
                        # Terminate process
                        terminate_result = crowdstrike.terminate_process(process['process_id'])
                        if terminate_result:
                            terminated_processes.append({
                                'process': process.get('process_name'),
                                'pid': process['process_id'],
                                'host': system['ip']
                            })
                            eradication_actions.append({
                                'action': 'process_termination',
                                'target': f"{process.get('process_name')} (PID: {process['process_id']}) on {system['ip']}",
                                'method': 'CrowdStrike',
                                'status': 'success',
                                'timestamp': eradication_time
                            })
                            print(f"   Terminated: {process.get('process_name')} on {system['ip']}")
except Exception as e:
    print(f"   Error terminating processes: {e}")

# 2. Remove malicious files and encryption tools
print(f"\n[ERADICATION] Removing malicious files and encryption tools...")
try:
    for system in affected_systems:
        if system.get('ip'):
            # Scan for and remove encryption-related malware
            malware_scan = crowdstrike.scan_and_remove_malware(system['ip'], 'encryption_tools')
            if malware_scan and malware_scan.get('removed_files'):
                for file_path in malware_scan['removed_files']:
                    deleted_files.append({
                        'file': file_path,
                        'host': system['ip']
                    })
                    eradication_actions.append({
                        'action': 'file_removal',
                        'target': f"{file_path} on {system['ip']}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Removed encryption tool: {file_path} on {system['ip']}")

    # Use Shuffle for additional cleanup of encryption artifacts
    for indicator in threat_indicators:
        if indicator.get('type') == 'file' and 'encrypt' in indicator['value'].lower():
            cleanup_result = shuffle.remove_malicious_file(indicator['value'])
            if cleanup_result:
                deleted_files.append({
                    'file': indicator['value'],
                    'host': 'multiple'
                })
                print(f"   Cleaned up encryption file: {indicator['value']}")
except Exception as e:
    print(f"   Error removing malicious files: {e}")

# 3. Reset compromised credentials
print(f"\n[ERADICATION] Resetting compromised credentials...")
try:
    for account in disabled_accounts:
        # Generate new password and reset account
        reset_result = shuffle.reset_account_password(account)
        if reset_result:
            reset_credentials.append(account)
            eradication_actions.append({
                'action': 'credential_reset',
                'target': account,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': eradication_time
            })
            print(f"   Reset credentials for: {account}")
except Exception as e:
    print(f"   Error resetting credentials: {e}")

# 4. Revoke suspicious certificates
print(f"\n[ERADICATION] Revoking suspicious certificates...")
try:
    revoked_certs = []
    for indicator in threat_indicators:
        if indicator.get('type') == 'suspicious_certificate':
            # Revoke certificate via Shuffle
            revoke_result = shuffle.revoke_certificate(indicator['value'])
            if revoke_result:
                revoked_certs.append(indicator['value'])
                eradication_actions.append({
                    'action': 'certificate_revocation',
                    'target': indicator['value'],
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Revoked certificate: {indicator['value']}")

    if revoked_certs:
        print(f"   Revoked {len(revoked_certs)} suspicious certificates")

except Exception as e:
    print(f"   Error revoking certificates: {e}")

# 5. Patch encryption-related vulnerabilities
print(f"\n[ERADICATION] Patching encryption-related vulnerabilities...")
try:
    # Identify common vulnerabilities exploited in encrypted channel attacks
    vulnerabilities = [
        'CVE-2022-40684',  # Fortinet SSL VPN vulnerability
        'CVE-2021-4034',   # Polkit privilege escalation
        'CVE-2021-44228',  # Log4j
        'CVE-2022-22965',  # Spring4Shell
        'CVE-2021-34527',  # PrintNightmare
    ]

    patched_systems = []
    for system in affected_systems:
        if system.get('ip'):
            # Check and patch vulnerabilities
            patch_result = shuffle.patch_system_vulnerabilities(system['ip'], vulnerabilities)
            if patch_result and patch_result.get('patched_count', 0) > 0:
                patched_systems.append(system['ip'])
                eradication_actions.append({
                    'action': 'vulnerability_patch',
                    'target': f"{system['ip']} ({patch_result.get('patched_count')} patches)",
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Patched {patch_result.get('patched_count')} vulnerabilities on {system['ip']}")

    if patched_systems:
        print(f"   Vulnerabilities patched on {len(patched_systems)} systems")

except Exception as e:
    print(f"   Error patching vulnerabilities: {e}")

# 6. Verify threat removal
print(f"\n[ERADICATION] Verifying threat removal...")
try:
    verification_results = []
    for system in affected_systems:
        if system.get('ip'):
            # Re-scan system for encrypted channel threats
            verification_scan = crowdstrike.verify_encrypted_channel_threats(system['ip'])
            verification_results.append({
                'system': system['ip'],
                'threats_remaining': verification_scan.get('encrypted_threats_found', 0),
                'scan_status': verification_scan.get('status', 'unknown')
            })

    clean_systems = [v for v in verification_results if v['threats_remaining'] == 0]
    eradication_actions.append({
        'action': 'threat_verification',
        'target': f"{len(clean_systems)}/{len(verification_results)} systems verified clean",
        'method': 'CrowdStrike',
        'status': 'success' if len(clean_systems) == len(verification_results) else 'partial',
        'timestamp': eradication_time
    })

    print(f"   Threat verification complete: {len(clean_systems)}/{len(verification_results)} systems clean")

except Exception as e:
    print(f"   Error verifying threat removal: {e}")

print(f"\n Eradication complete:")
print(f"  - Processes terminated: {len(terminated_processes)}")
print(f"  - Files removed: {len(deleted_files)}")
print(f"  - Credentials reset: {len(reset_credentials)}")
print(f"  - Certificates revoked: {len(revoked_certs) if 'revoked_certs' in locals() else 0}")
print(f"  - Systems verified clean: {len([v for v in verification_results if v.get('threats_remaining', 0) == 0])}")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
reenabled_hosts = []
restored_accounts = []

# 1. Re-enable isolated systems
print(f"\n[RECOVERY] Re-enabling isolated systems...")
try:
    for host in isolated_hosts:
        # Use CrowdStrike to re-enable host
        reenable_result = crowdstrike.reenable_host_by_ip(host)
        if reenable_result and reenable_result.get('status') == 'reenabled':
            reenabled_hosts.append(host)
            recovery_actions.append({
                'action': 'host_reenable',
                'target': host,
                'method': 'CrowdStrike',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Re-enabled host: {host}")
        else:
            # Use Shuffle for network re-enablement
            network_restore = shuffle.restore_system(host)
            if network_restore:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'network_restore',
                    'target': host,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Restored network access: {host}")
except Exception as e:
    print(f"   Error re-enabling systems: {e}")

# 2. Restore disabled accounts
print(f"\n[RECOVERY] Restoring disabled accounts...")
try:
    for user in disabled_accounts:
        # Restore account access via Shuffle
        restore_result = shuffle.restore_account(user)
        if restore_result:
            restored_accounts.append(user)
            recovery_actions.append({
                'action': 'account_restore',
                'target': user,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored account: {user}")
except Exception as e:
    print(f"   Error restoring accounts: {e}")

# 3. Validate system functionality
print(f"\n[RECOVERY] Validating system functionality...")
try:
    for system in affected_systems:
        if system.get('ip'):
            # Validate system health after recovery
            health_check = crowdstrike.validate_system_health_by_ip(system['ip'])
            if health_check and health_check.get('status') == 'healthy':
                recovery_actions.append({
                    'action': 'system_validation',
                    'target': system['ip'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   System health validated: {system['ip']}")
            else:
                print(f"   System health issues detected: {system['ip']}")
except Exception as e:
    print(f"   Error validating system health: {e}")

# 4. Restore monitoring and alerting
print(f"\n[RECOVERY] Restoring monitoring and alerting...")
try:
    # Restore normal Splunk monitoring rules
    normal_monitoring = splunk.restore_normal_monitoring()
    if normal_monitoring:
        recovery_actions.append({
            'action': 'monitoring_restore',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal monitoring in Splunk")

    # Restore CrowdStrike normal operations
    cs_normal_ops = crowdstrike.restore_normal_operations()
    if cs_normal_ops:
        recovery_actions.append({
            'action': 'cs_normal_ops',
            'target': 'CrowdStrike',
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal CrowdStrike operations")

except Exception as e:
    print(f"   Error restoring monitoring: {e}")

# 5. Verify data integrity and encryption status
print(f"\n[RECOVERY] Verifying data integrity and encryption status...")
try:
    integrity_checks = []
    for system in affected_systems:
        if system.get('ip'):
            # Check that no unauthorized encrypted connections occurred during incident
            integrity_result = shuffle.verify_no_unauthorized_encrypted_connections(system['ip'])
            integrity_checks.append({
                'system': system['ip'],
                'integrity_ok': integrity_result.get('no_unauthorized_encrypted', True),
                'connections_detected': integrity_result.get('recent_encrypted_connections', 0)
            })

    systems_with_integrity = [c for c in integrity_checks if c['integrity_ok']]
    recovery_actions.append({
        'action': 'data_integrity_check',
        'target': f"{len(systems_with_integrity)}/{len(integrity_checks)} systems with verified integrity",
        'method': 'Shuffle',
        'status': 'success' if len(systems_with_integrity) == len(integrity_checks) else 'partial',
        'timestamp': recovery_time
    })

    print(f"   Data integrity check complete: {len(systems_with_integrity)}/{len(integrity_checks)} systems OK")

except Exception as e:
    print(f"   Error verifying data integrity: {e}")

print(f"\n Recovery complete:")
print(f"  - Hosts re-enabled: {len(reenabled_hosts)}")
print(f"  - Accounts restored: {len(restored_accounts)}")
print(f"  - Systems validated: ")
print(f"  - Data integrity verified: {len([c for c in integrity_checks if c.get('integrity_ok', False)])} systems")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_time = datetime.now().isoformat()
post_incident_actions = []

# 1. Generate comprehensive incident report
print(f"\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'T1573 - Encrypted Channel',
        'detection_time': detection_time,
        'containment_time': containment_time,
        'eradication_time': eradication_time,
        'recovery_time': recovery_time,
        'post_incident_time': post_incident_time,
        'affected_systems': len(affected_systems),
        'isolated_hosts': len(isolated_hosts),
        'blocked_connections': len(blocked_connections),
        'disabled_accounts': len(disabled_accounts),
        'terminated_processes': len(terminated_processes),
        'deleted_files': len(deleted_files),
        'reset_credentials': len(reset_credentials),
        'revoked_certificates': len(revoked_certs) if 'revoked_certs' in locals() else 0,
        'reenabled_hosts': len(reenabled_hosts),
        'restored_accounts': len(restored_accounts),
        'threat_indicators': len(threat_indicators),
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': post_incident_time
        },
        'actions_taken': {
            'detection': detection_actions,
            'containment': containment_actions,
            'eradication': eradication_actions,
            'recovery': recovery_actions
        },
        'recommendations': [
            'Implement stricter encrypted channel monitoring',
            'Deploy SSL/TLS inspection capabilities',
            'Regular certificate validation and management',
            'Enhance network traffic analysis for encrypted protocols',
            'Conduct user awareness training on encrypted C2 indicators'
        ]
    }

    # Save report to IRIS
    iris_report = iris.create_incident_report(incident_report)
    if iris_report:
        post_incident_actions.append({
            'action': 'incident_report',
            'target': 'IRIS',
            'method': 'IRIS',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Incident report saved to IRIS: {iris_report.get('report_id', 'N/A')}")

    # Share threat intelligence with MISP
    misp_sharing = misp.share_threat_intelligence(threat_indicators, incident_report)
    if misp_sharing:
        post_incident_actions.append({
            'action': 'threat_intel_sharing',
            'target': 'MISP',
            'method': 'MISP',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Threat intelligence shared with MISP community")

except Exception as e:
    print(f"   Error generating incident report: {e}")

# 2. Conduct lessons learned analysis
print(f"\n[POST-INCIDENT] Conducting lessons learned analysis...")
try:
    lessons_learned = {
        'incident_type': 'Encrypted Channel C2 Activity (T1573)',
        'root_cause': 'Insufficient monitoring of encrypted network communications for C2 detection',
        'detection_effectiveness': 'High - Automated detection via Splunk and CrowdStrike',
        'response_effectiveness': 'High - Automated containment and eradication successful',
        'recovery_effectiveness': 'High - Systems restored with integrity verification',
        'gaps_identified': [
            'Need enhanced SSL/TLS traffic inspection capabilities',
            'Consider implementing certificate pinning and validation',
            'Improve encrypted protocol anomaly detection'
        ],
        'improvements_needed': [
            'Update detection rules for new encrypted C2 techniques',
            'Implement automated response for encrypted channel abuse',
            'Enhance certificate management and monitoring'
        ],
        'preventive_measures': [
            'Deploy SSL/TLS inspection and decryption',
            'Implement certificate transparency monitoring',
            'Regular security assessments for encrypted protocols',
            'Enhanced user training on encrypted C2 risks'
        ]
    }

    # Document lessons learned in IRIS
    lessons_doc = iris.document_lessons_learned(lessons_learned)
    if lessons_doc:
        post_incident_actions.append({
            'action': 'lessons_learned',
            'target': 'IRIS',
            'method': 'IRIS',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Lessons learned documented in IRIS")

except Exception as e:
    print(f"   Error conducting lessons learned: {e}")

# 3. Implement preventive measures
print(f"\n[POST-INCIDENT] Implementing preventive measures...")
try:
    preventive_actions = []

    # Update Splunk monitoring rules
    enhanced_monitoring = splunk.implement_enhanced_monitoring('encrypted_channel_c2')
    if enhanced_monitoring:
        preventive_actions.append('Enhanced Splunk monitoring for encrypted channels')
        print(f"   Enhanced Splunk monitoring rules implemented")

    # Update CrowdStrike policies
    policy_updates = crowdstrike.update_policies('encrypted_channel_protection')
    if policy_updates:
        preventive_actions.append('Updated CrowdStrike endpoint policies')
        print(f"   CrowdStrike policies updated for encrypted channel protection")

    # Implement Shuffle automated responses
    auto_response = shuffle.implement_automated_response('encrypted_channel_c2')
    if auto_response:
        preventive_actions.append('Automated response workflows implemented')
        print(f"   Automated response workflows implemented in Shuffle")

    post_incident_actions.append({
        'action': 'preventive_measures',
        'target': preventive_actions,
        'method': 'Multiple',
        'status': 'success',
        'timestamp': post_incident_time
    })

except Exception as e:
    print(f"   Error implementing preventive measures: {e}")

# 4. Update security policies and procedures
print(f"\n[POST-INCIDENT] Updating security policies...")
try:
    policy_updates = {
        'encrypted_channel_policy': {
            'description': 'Enhanced encrypted channel monitoring and control',
            'changes': [
                'Added monitoring for suspicious encrypted communications',
                'Implemented SSL/TLS inspection capabilities',
                'Enhanced certificate validation and management',
                'Added automated response for encrypted channel anomalies'
            ]
        },
        'incident_response_procedure': {
            'description': 'Updated IR procedures for encrypted channel C2 incidents',
            'changes': [
                'Added automated containment steps for encrypted connections',
                'Enhanced eradication procedures for encryption-related threats',
                'Improved recovery validation for encrypted channel incidents'
            ]
        }
    }

    # Update policies in Shuffle (workflow management)
    policy_update_result = shuffle.update_security_policies(policy_updates)
    if policy_update_result:
        post_incident_actions.append({
            'action': 'policy_updates',
            'target': 'Shuffle',
            'method': 'Shuffle',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Security policies updated in Shuffle")

except Exception as e:
    print(f"   Error updating security policies: {e}")

# 5. Close incident case
print(f"\n[POST-INCIDENT] Closing incident case...")
try:
    closure_summary = {
        'incident_id': incident_id,
        'closure_time': post_incident_time,
        'resolution_status': 'Resolved - Encrypted channel threat contained and eradicated',
        'final_status': 'Closed',
        'total_actions': len(detection_actions) + len(containment_actions) + len(eradication_actions) + len(recovery_actions) + len(post_incident_actions),
        'affected_assets': len(affected_systems),
        'threat_actor': 'Unknown - Automated response prevented further encrypted C2 activity',
        'data_compromised': 'Encrypted channel traffic - contained before data exfiltration',
        'business_impact': 'Minimal - Automated response prevented encrypted C2 establishment'
    }

    # Close case in IRIS
    case_closure = iris.close_incident_case(incident_id, closure_summary)
    if case_closure:
        post_incident_actions.append({
            'action': 'case_closure',
            'target': f"IRIS Case {incident_id}",
            'method': 'IRIS',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Incident case closed in IRIS: {incident_id}")

    # Archive incident data
    archive_result = shuffle.archive_incident_data(incident_id)
    if archive_result:
        print(f"   Incident data archived for compliance and analysis")

except Exception as e:
    print(f"   Error closing incident case: {e}")

print(f"\n Post-incident actions complete:")
print(f"  - Incident report generated: ")
print(f"  - Lessons learned documented: ")
print(f"  - Preventive measures implemented: ")
print(f"  - Security policies updated: ")
print(f"  - Incident case closed: ")
print(f"\n Encrypted Channel C2 Incident Response Complete")
print(f"   Incident ID: {incident_id}")
print(f"   Total Response Time: Automated (minutes)")
print(f"   Systems Protected: {len(affected_systems)}")
print(f"   Threat Contained: ")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
